In [1]:
%load_ext autoreload
%autoreload 2
from src.data_loading import load_data, PROCESSED_DATA_PATH
import pandas as pd
from sklearn.metrics import roc_auc_score
from src.evaluation import *
from src.training import run_lgbm_cv
import src.secondary_tables as st
application_train, application_test = load_data()
y = application_train[TARGET_COL]

auc_results = {}

def record_auc(stage, X, results):
    auc_results[stage] = {
        "num_features": X.shape[1],
        "mean_auc": results["mean_auc"],
        "std_auc": results["std_auc"],
        "oof_auc": roc_auc_score(y, results["oof_preds"]),
    }
    auc_table = pd.DataFrame.from_dict(auc_results, orient="index").reset_index()
    return auc_table.rename(columns={"index": "stage"})


In [2]:
from src.data_loading import ROOT, RAW_DATA_PATH
bureau = pd.read_csv(RAW_DATA_PATH / 'bureau.csv')
application_train_1, application_test_1 = st.add_bureau(application_train, application_test, bureau)
X_1 = application_train_1[[c for c in application_train_1.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_1, y)
summarize_cv_results(results, y)
record_auc("bureau", X_1, results)

mean_auc: 0.7682222671587162
std_auc: 0.0035437528228860473
oof_auc: 0.7682062373049657


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.768222,0.003544,0.768206


In [4]:
bureau = pd.read_csv(RAW_DATA_PATH / 'bureau.csv')
bureau = st.add_bureau_balance(bureau)
application_train_2, application_test_2 = st.add_bureau(application_train, application_test, bureau)
X_2 = application_train_2[[c for c in application_train_2.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_2, y)
summarize_cv_results(results, y)
application_train_2.to_csv(PROCESSED_DATA_PATH / 'application_train_2.csv')
application_test_2.to_csv(PROCESSED_DATA_PATH / 'application_test_2.csv')
record_auc("bureau + bureau_balance", X_2, results)

mean_auc: 0.7680341429065892
std_auc: 0.003858951849597465
oof_auc: 0.7680093292085799


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.768222,0.003544,0.768206
1,bureau + bureau_balance,227,0.768034,0.003859,0.768009


In [5]:
application_train_3, application_test_3 = st.add_previous_application(application_train_2, application_test_2)
X_3 = application_train_3[[c for c in application_train_3.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_3, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application", X_3, results)

mean_auc: 0.7761920785544218
std_auc: 0.003865436706647357
oof_auc: 0.7761583503106322


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.768222,0.003544,0.768206
1,bureau + bureau_balance,227,0.768034,0.003859,0.768009
2,bureau + bureau_balance + previous_application,362,0.776192,0.003865,0.776158


In [6]:
application_train_4, application_test_4 = st.add_pos_cash_balance(application_train_3, application_test_3)
X_4 = application_train_4[[c for c in application_train_4.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_4, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application + POS_CASH_balance", X_4, results)

mean_auc: 0.7780734683657142
std_auc: 0.003958455920872187
oof_auc: 0.7780658007921835


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.768222,0.003544,0.768206
1,bureau + bureau_balance,227,0.768034,0.003859,0.768009
2,bureau + bureau_balance + previous_application,362,0.776192,0.003865,0.776158
3,bureau + bureau_balance + previous_application...,399,0.778073,0.003958,0.778066


In [7]:
application_train_5, application_test_5 = st.add_credit_card_balance(application_train_4, application_test_4)
X_5 = application_train_5[[c for c in application_train_5.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_5, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application + POS_CASH_balance + credit_card_balance", X_5, results)

mean_auc: 0.779503220366443
std_auc: 0.0030589450290355895
oof_auc: 0.7794893028713854


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.768222,0.003544,0.768206
1,bureau + bureau_balance,227,0.768034,0.003859,0.768009
2,bureau + bureau_balance + previous_application,362,0.776192,0.003865,0.776158
3,bureau + bureau_balance + previous_application...,399,0.778073,0.003958,0.778066
4,bureau + bureau_balance + previous_application...,522,0.779503,0.003059,0.779489


In [8]:
application_train_6, application_test_6 = st.add_installments_payments(application_train_5, application_test_5)
application_train_6.to_csv(PROCESSED_DATA_PATH / 'application_train_with_secondary_tables.csv')
application_test_6.to_csv(PROCESSED_DATA_PATH / 'application_test_with_secondary_tables.csv')
X_6 = application_train_6[[c for c in application_train_6.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_6, y)
summarize_cv_results(results, y)
record_auc("all secondary tables", X_6, results)

mean_auc: 0.7826488084234313
std_auc: 0.0037920963870830523
oof_auc: 0.7826313938697076


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.768222,0.003544,0.768206
1,bureau + bureau_balance,227,0.768034,0.003859,0.768009
2,bureau + bureau_balance + previous_application,362,0.776192,0.003865,0.776158
3,bureau + bureau_balance + previous_application...,399,0.778073,0.003958,0.778066
4,bureau + bureau_balance + previous_application...,522,0.779503,0.003059,0.779489
5,all secondary tables,568,0.782649,0.003792,0.782631
